In [1]:
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Tuple, Union, Optional, FrozenSet
import functools
import math
import random
from pettingzoo import ParallelEnv
from gymnasium import spaces

In [2]:
# =====================================================================
# Helper Types & Converters for Hashable States
# =====================================================================

def freeze(d: Dict[str, Any]) -> FrozenSet[Tuple[str, Any]]:
    """Converts a dictionary into a hashable frozenset of key-value pairs."""
    return frozenset(d.items())

def unfreeze(fs: FrozenSet[Tuple[str, Any]]) -> Dict[str, Any]:
    """Converts a frozenset back into a standard dictionary."""
    return dict(fs)

In [3]:
# =====================================================================
# 1. POSG Abstract Base Class
# =====================================================================

class AbstractPOSG(ABC):
    """
    Abstract Base Class for Partially Observable Stochastic Games (POSG).
    Supports hashable states/observations, explicit metadata, action catalogues,
    validation, schema generation, and general stochastic observation modeling.
    """

    @property
    @abstractmethod
    def players(self) -> Dict[str, Dict[str, Any]]:
        """
        Player metadata mapping player_id to descriptions/properties.
        Example: {"player_1": {"description": "First player"}}
        """
        pass

    @property
    @abstractmethod
    def state_variables(self) -> Dict[str, Dict[str, Any]]:
        """
        State-variable metadata.
        Example: {"phase": {"type": "str", "description": "Current game phase"}}
        """
        pass

    @property
    @abstractmethod
    def action_catalogue(self) -> Dict[str, Dict[str, Any]]:
        """
        Catalogue of actions and their descriptions.
        Example: {"rock": {"description": "Play rock"}}
        """
        pass

    @property
    @abstractmethod
    def initial_state(self) -> FrozenSet[Tuple[str, Any]]:
        """Returns the initial state of the game as a hashable frozenset."""
        pass

    @abstractmethod
    def legal_actions(self, state: FrozenSet[Tuple[str, Any]], player: str) -> List[str]:
        """Returns list of legal action strings for a given player and state."""
        pass

    @abstractmethod
    def transition_kernel(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str]
    ) -> Dict[Tuple[FrozenSet[Tuple[str, Any]], FrozenSet[Tuple[str, Any]]], float]:
        """
        General Stochastic Kernel: P(s', o | s, a)
        Returns mapping: (next_state, observation_profile) -> probability
        Both next_state and observation_profile are hashable frozensets.
        """
        pass

    @abstractmethod
    def rewards(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str], next_state: FrozenSet[Tuple[str, Any]]
    ) -> Dict[str, float]:
        """Returns rewards for each player."""
        pass

    @abstractmethod
    def is_terminal(self, state: FrozenSet[Tuple[str, Any]]) -> bool:
        """Returns True if state is terminal."""
        pass

    # --- Validation Mechanism ---
    def validate(self) -> List[str]:
        """
        Reliability mechanism for verifying POSG implementations.
        Catches unhashable states, invalid actions, probability sum errors, and bad metadata.
        """
        errors = []

        # 1. Declarative Metadata Checks
        if not isinstance(self.players, dict) or not self.players:
            errors.append("Validation Error: 'players' must be a non-empty dictionary.")
        if not isinstance(self.state_variables, dict):
            errors.append("Validation Error: 'state_variables' must be a dictionary.")
        if not isinstance(self.action_catalogue, dict) or not self.action_catalogue:
            errors.append("Validation Error: 'action_catalogue' must be a non-empty dictionary.")

        # 2. Initial State Verification
        try:
            init_s = self.initial_state
            hash(init_s)  # Verify hashability
        except Exception as e:
            errors.append(f"Validation Error: Initial state is invalid or unhashable: {e}")
            return errors

        # 3. Reachability & Kernel Distribution Checks
        visited_states = set()
        queue = [init_s]

        while queue:
            curr_state = queue.pop(0)
            if curr_state in visited_states:
                continue
            visited_states.add(curr_state)

            if self.is_terminal(curr_state):
                continue

            # Validate legal actions
            player_actions = {}
            for p in self.players.keys():
                leg_acts = self.legal_actions(curr_state, p)
                if not leg_acts:
                    errors.append(f"Validation Error: Player '{p}' has no legal actions in state {dict(curr_state)}.")
                for act in leg_acts:
                    if act not in self.action_catalogue:
                        errors.append(f"Validation Error: Undeclared action '{act}' for player '{p}'.")
                player_actions[p] = leg_acts

            # Sample action profile to validate kernel distribution
            if len(player_actions) == len(self.players):
                sample_joint_action = {p: acts[0] for p, acts in player_actions.items() if acts}
                if len(sample_joint_action) == len(self.players):
                    try:
                        dist = self.transition_kernel(curr_state, sample_joint_action)
                        prob_sum = sum(dist.values())
                        
                        if not math.isclose(prob_sum, 1.0, abs_tol=1e-5):
                            errors.append(f"Validation Error: Probabilities sum to {prob_sum} (expected 1.0) for state {dict(curr_state)}.")

                        for (next_s, obs_prof), prob in dist.items():
                            hash(next_s)
                            hash(obs_prof)
                            
                            if prob < 0.0 or prob > 1.0:
                                errors.append(f"Validation Error: Invalid probability {prob} in transition distribution.")
                            
                            obs_dict = dict(obs_prof)
                            if set(obs_dict.keys()) != set(self.players.keys()):
                                errors.append("Validation Error: Observation profile players mismatch declared players.")
                            
                            if next_s not in visited_states:
                                queue.append(next_s)
                    except Exception as e:
                        errors.append(f"Validation Error during kernel execution at state {dict(curr_state)}: {e}")

        return errors

    # --- Schema Generation ---
    def schema(self) -> str:
        """Generates a human/LLM-readable structural schema of the game."""
        lines = ["=== POSG Game Schema ===", "\nPlayers:"]
        for p, meta in self.players.items():
            lines.append(f"  - {p}: {meta.get('description', 'No description')}")

        lines.append("\nState Variables:")
        for sv, meta in self.state_variables.items():
            lines.append(f"  - {sv} ({meta.get('type', 'Any')}): {meta.get('description', '')}")

        lines.append("\nAction Catalogue:")
        for act, meta in self.action_catalogue.items():
            lines.append(f"  - {act}: {meta.get('description', '')}")

        lines.append(f"\nInitial State: {dict(self.initial_state)}")
        return "\n".join(lines)

In [4]:
# =====================================================================
# 2. Reference Implementation G.1: Rock-Paper-Scissors (RPS)
# =====================================================================

class RockPaperScissorsPOSG(AbstractPOSG):
    """G.1 Reference Implementation using explicit metadata and hashable state kernels."""

    @property
    def players(self) -> Dict[str, Dict[str, Any]]:
        return {
            "player_1": {"description": "First agent choosing Rock, Paper, or Scissors"},
            "player_2": {"description": "Second agent choosing Rock, Paper, or Scissors"}
        }

    @property
    def state_variables(self) -> Dict[str, Dict[str, Any]]:
        return {
            "phase": {"type": "str", "description": "Current phase of the game ('start' or 'terminal')"},
            "winner": {"type": "Optional[str]", "description": "Winner ID or 'draw'"}
        }

    @property
    def action_catalogue(self) -> Dict[str, Dict[str, Any]]:
        return {
            "rock": {"description": "Play Rock (beats Scissors, loses to Paper)"},
            "paper": {"description": "Play Paper (beats Rock, loses to Scissors)"},
            "scissors": {"description": "Play Scissors (beats Paper, loses to Rock)"}
        }

    @property
    def initial_state(self) -> FrozenSet[Tuple[str, Any]]:
        return freeze({"phase": "start", "winner": None})

    def legal_actions(self, state: FrozenSet[Tuple[str, Any]], player: str) -> List[str]:
        if self.is_terminal(state):
            return []
        return ["rock", "paper", "scissors"]

    def transition_kernel(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str]
    ) -> Dict[Tuple[FrozenSet[Tuple[str, Any]], FrozenSet[Tuple[str, Any]]], float]:
        if self.is_terminal(state):
            term_obs = freeze({"player_1": "game_over", "player_2": "game_over"})
            return {(state, term_obs): 1.0}

        a1, a2 = actions["player_1"], actions["player_2"]
        if a1 == a2:
            winner = "draw"
        elif (a1 == "rock" and a2 == "scissors") or \
             (a1 == "paper" and a2 == "rock") or \
             (a1 == "scissors" and a2 == "paper"):
            winner = "player_1"
        else:
            winner = "player_2"

        next_state = freeze({"phase": "terminal", "winner": winner})
        obs_profile = freeze({
            "player_1": f"opp_played_{a2}",
            "player_2": f"opp_played_{a1}"
        })
        
        return {(next_state, obs_profile): 1.0}

    def rewards(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str], next_state: FrozenSet[Tuple[str, Any]]
    ) -> Dict[str, float]:
        st_dict = unfreeze(next_state)
        winner = st_dict.get("winner")
        if winner == "player_1":
            return {"player_1": 1.0, "player_2": -1.0}
        elif winner == "player_2":
            return {"player_1": -1.0, "player_2": 1.0}
        return {"player_1": 0.0, "player_2": 0.0}

    def is_terminal(self, state: FrozenSet[Tuple[str, Any]]) -> bool:
        st_dict = unfreeze(state)
        return st_dict.get("phase") == "terminal"

In [5]:
# =====================================================================
# 3. Reference Implementation G.2: Dice Duel (Stochastic POSG)
# =====================================================================

class DiceDuelPOSG(AbstractPOSG):
    """
    G.2 Reference Implementation demonstrating stochastic state transitions and
    partial observability via dice rolling mechanics.
    """

    @property
    def players(self) -> Dict[str, Dict[str, Any]]:
        return {
            "player_1": {"description": "Player 1"},
            "player_2": {"description": "Player 2"}
        }

    @property
    def state_variables(self) -> Dict[str, Dict[str, Any]]:
        return {
            "phase": {"type": "str", "description": "'roll' or 'terminal'"},
            "result": {"type": "str", "description": "Outcome description"}
        }

    @property
    def action_catalogue(self) -> Dict[str, Dict[str, Any]]:
        return {
            "roll": {"description": "Roll a 6-sided die"}
        }

    @property
    def initial_state(self) -> FrozenSet[Tuple[str, Any]]:
        return freeze({"phase": "roll", "result": "pending"})

    def legal_actions(self, state: FrozenSet[Tuple[str, Any]], player: str) -> List[str]:
        if self.is_terminal(state):
            return []
        return ["roll"]

    def transition_kernel(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str]
    ) -> Dict[Tuple[FrozenSet[Tuple[str, Any]], FrozenSet[Tuple[str, Any]]], float]:
        if self.is_terminal(state):
            term_obs = freeze({"player_1": "over", "player_2": "over"})
            return {(state, term_obs): 1.0}

        dist = {}
        prob = 1.0 / 36.0
        for d1 in range(1, 7):
            for d2 in range(1, 7):
                if d1 > d2:
                    w = "player_1"
                elif d2 > d1:
                    w = "player_2"
                else:
                    w = "draw"

                next_s = freeze({"phase": "terminal", "result": w})
                obs = freeze({"player_1": f"rolled_{d1}", "player_2": f"rolled_{d2}"})
                key = (next_s, obs)
                dist[key] = dist.get(key, 0.0) + prob
        return dist

    def rewards(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str], next_state: FrozenSet[Tuple[str, Any]]
    ) -> Dict[str, float]:
        st = unfreeze(next_state)
        res = st.get("result")
        if res == "player_1":
            return {"player_1": 1.0, "player_2": -1.0}
        elif res == "player_2":
            return {"player_1": -1.0, "player_2": 1.0}
        return {"player_1": 0.0, "player_2": 0.0}

    def is_terminal(self, state: FrozenSet[Tuple[str, Any]]) -> bool:
        return unfreeze(state).get("phase") == "terminal"

In [6]:
# =====================================================================
# 4. Thin PettingZoo ParallelEnv Wrapper
# =====================================================================

class POSGParallelEnv(ParallelEnv):
    """PettingZoo ParallelEnv wrapper around AbstractPOSG game engines."""

    metadata = {"render_modes": ["human"], "name": "posg_wrapper_v0"}

    def __init__(self, posg_game: AbstractPOSG):
        super().__init__()
        self.game = posg_game

        # Validate game design upon environment instantiation
        errors = self.game.validate()
        if errors:
            raise ValueError(f"POSG Validation failed:\n" + "\n".join(errors))

        self.possible_agents = list(self.game.players.keys())
        self.agents = self.possible_agents.copy()

        # Build discrete spaces from action catalogue
        action_list = list(self.game.action_catalogue.keys())
        self._action_to_int = {act: idx for idx, act in enumerate(action_list)}
        self._int_to_action = {idx: act for idx, act in enumerate(action_list)}

        self._action_spaces = {
            agent: spaces.Discrete(len(action_list)) for agent in self.possible_agents
        }
        self._observation_spaces = {
            agent: spaces.Text(max_length=128) for agent in self.possible_agents
        }

        self.current_state = None

    @functools.lru_cache(maxsize=None)
    def observation_space(self, agent: str):
        return self._observation_spaces[agent]

    @functools.lru_cache(maxsize=None)
    def action_space(self, agent: str):
        return self._action_spaces[agent]

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None):
        if seed is not None:
            random.seed(seed)
        self.agents = self.possible_agents.copy()
        self.current_state = self.game.initial_state

        observations = {agent: "initial_observation" for agent in self.agents}
        infos = {agent: {} for agent in self.agents}
        return observations, infos

    def step(self, actions: Dict[str, int]):
        if not self.agents:
            return {}, {}, {}, {}, {}

        # Convert discrete action indices to declarative action strings
        string_actions = {
            agent: self._int_to_action[act_id]
            for agent, act_id in actions.items()
            if agent in self.agents
        }

        # Sample next state & observation profile from kernel P(s', o | s, a)
        dist = self.game.transition_kernel(self.current_state, string_actions)
        outcomes, probs = list(dist.keys()), list(dist.values())
        selected_idx = random.choices(range(len(outcomes)), weights=probs, k=1)[0]
        next_state, obs_profile = outcomes[selected_idx]

        # Rewards & status
        rewards = self.game.rewards(self.current_state, string_actions, next_state)
        is_term = self.game.is_terminal(next_state)
        self.current_state = next_state

        obs_dict = unfreeze(obs_profile)
        terminations = {agent: is_term for agent in self.agents}
        truncations = {agent: False for agent in self.agents}
        infos = {agent: {} for agent in self.agents}

        if is_term:
            self.agents = []

        return obs_dict, rewards, terminations, truncations, infos

In [7]:
# =====================================================================
# 5. Execution Pipeline
# =====================================================================

if __name__ == "__main__":
    print("=== Testing G.1 Rock-Paper-Scissors ===")
    rps_game = RockPaperScissorsPOSG()

    # 1. Print Declarative Schema
    print("\n" + rps_game.schema())

    # 2. Run Validation Engine
    errors = rps_game.validate()
    print(f"\nValidation Result: {'PASSED ✅' if not errors else 'FAILED ❌'}")
    if errors:
        for err in errors:
            print(f"  - {err}")

    # 3. Test PettingZoo Execution
    env = POSGParallelEnv(rps_game)
    obs, infos = env.reset(seed=42)
    print(f"\nEnv Reset Observations: {obs}")

    # Step: Player 1 (Rock=0), Player 2 (Scissors=2)
    joint_action = {"player_1": 0, "player_2": 2}
    obs, rews, terms, truncs, infos = env.step(joint_action)

    print(f"\nAction Executed: Rock vs Scissors")
    print(f"Observations Received: {obs}")
    print(f"Rewards Assigned: {rews}")
    print(f"Is Terminated: {terms}")

    print("\n=== Testing G.2 Dice Duel (Stochastic Game) ===")
    dice_game = DiceDuelPOSG()
    print("\nDice Duel Schema Preview:")
    print(dice_game.schema())
    
    print("\nValidation Result:", "PASSED ✅" if not dice_game.validate() else "FAILED ❌")

    env_dice = POSGParallelEnv(dice_game)
    obs, infos = env_dice.reset(seed=42)
    obs, rews, terms, truncs, infos = env_dice.step({"player_1": 0, "player_2": 0})
    print(f"\nDice Duel Step Observations: {obs}")
    print(f"Dice Duel Step Rewards: {rews}")

=== Testing G.1 Rock-Paper-Scissors ===

=== POSG Game Schema ===

Players:
  - player_1: First agent choosing Rock, Paper, or Scissors
  - player_2: Second agent choosing Rock, Paper, or Scissors

State Variables:
  - phase (str): Current phase of the game ('start' or 'terminal')
  - winner (Optional[str]): Winner ID or 'draw'

Action Catalogue:
  - rock: Play Rock (beats Scissors, loses to Paper)
  - paper: Play Paper (beats Rock, loses to Scissors)
  - scissors: Play Scissors (beats Paper, loses to Rock)

Initial State: {'winner': None, 'phase': 'start'}

Validation Result: PASSED ✅

Env Reset Observations: {'player_1': 'initial_observation', 'player_2': 'initial_observation'}

Action Executed: Rock vs Scissors
Observations Received: {'player_1': 'opp_played_scissors', 'player_2': 'opp_played_rock'}
Rewards Assigned: {'player_1': 1.0, 'player_2': -1.0}
Is Terminated: {'player_1': True, 'player_2': True}

=== Testing G.2 Dice Duel (Stochastic Game) ===

Dice Duel Schema Preview:
=== 